# Track A — 요양수요가속도 회귀 분석
**머신러닝 강의 프로젝트: 고령 취약계층 헬스케어 수요 예측**

| 항목 | 내용 |
|------|------|
| 분석 유형 | 지도학습 — 회귀 (Supervised Regression) |
| 타겟(Y) | 요양수요가속도 = `ln(독거노인수_2023 / 독거노인수_2018) × 100` |
| 데이터 | `df_analysis_v6.csv` — 420개 행정동, X 9개 피처 |
| 모델 | Ridge · Lasso · Decision Tree · Random Forest · XGBoost |
| 분석 순서 | ① 8:2 분할 후 기본 모델 Train/Test 비교 → ② CV (KFold·StratifiedKFold) → ③ GridSearchCV 튜닝 |
| CV 전략 | KFold(5) · StratifiedKFold(5, Y-분위수 층화) |
| 튜닝 방식 | 전 모델 GridSearchCV |
| 평가 지표 | RMSE · MAE · R² · MAPE |


---
## 0. 환경 설정 및 데이터 로드


In [16]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import matplotlib.font_manager as fm

from sklearn.base import clone
from sklearn.model_selection import (
    KFold, StratifiedKFold,
    cross_validate, GridSearchCV,
    learning_curve, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, make_scorer
)
from xgboost import XGBRegressor


# 한글 폰트
KOREAN_FONTS = [
    '/System/Library/Fonts/AppleSDGothicNeo.ttc',   # 가장 선호
    '/System/Library/Fonts/Supplemental/AppleGothic.ttf',
]
for fpath in KOREAN_FONTS:
    import os
    if os.path.exists(fpath):
        fm.fontManager.addfont(fpath)
        plt.rcParams['font.family'] = fm.FontProperties(fname=fpath).get_name()
        break

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120


# 코드 재현성
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('라이브러리 로드 완료 ✓')


라이브러리 로드 완료 ✓


In [17]:
# 데이터 로드
df = pd.read_csv('data/df_analysis_v6.csv', encoding='utf-8-sig')

ID_COLS = [
    '행정동코드', '행정동명', '시군구명', '자치구명',
    '연도', '총인구수', '65세이상인구', '독거노인수_2018'
]

FEAT_COLS = [
    '고령화율',        
    '고령화율_변화',   
    '독거노인비율',    # 가장 높은 상관 관계를 보인 피쳐
    '비율_75-79세',   
    '비율_85세이상',  
    '인프라갭',        
    '포화도',          
    '시설수',       
    '수요공급갭지수', 
]

TARGET = '요양수요가속도'

X_all = df[FEAT_COLS].copy()
y_all = df[TARGET].copy()
meta  = df[ID_COLS].copy()

print(f'피처 수: {len(FEAT_COLS)}개')
print(f'샘플 수: {len(X_all)}개 행정동')
print(f'\n[Y 기초 통계]')
print(y_all.describe().to_string())

neg_cnt = sum(1 for y in y_all if y < 0)
print(f'\n타겟 변수 중 음수의 개수 : {neg_cnt}')


피처 수: 9개
샘플 수: 420개 행정동

[Y 기초 통계]
count    420.000000
mean      30.197357
std       20.941252
min      -24.336000
25%       19.397500
50%       28.825000
75%       38.175000
max      114.752000

타겟 변수 중 음수의 개수 : 16


---
## 2. 데이터 분할 (8:2 Train/Test Split)


In [21]:
# Y를 5분위 구간으로 나눠 train/test에 분포를 균등하게 배분
y_bins_all = pd.qcut(y_all, q=5, labels=False, duplicates='drop')

X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X_all, y_all, meta,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_bins_all # 타겟변수의 불균형 방지
)

# 층화 CV에서 사용할 Y 분위 구간 (훈련 세트 기준)
y_train_bins = pd.qcut(y_train, q=5, labels=False, duplicates='drop')

# 인덱스 재설정
for df_part in [X_train, X_test, y_train, y_test, meta_train, meta_test, y_train_bins]:
    df_part.reset_index(drop=True, inplace=True)

# 데이터 분할 확인 
print(f'훈련 세트: {len(X_train)}개 행정동')
print(f'테스트 세트: {len(X_test)}개 행정동')
# 분할된 타겟 변수 분표 비교
print(f'\nTrain — 평균: {y_train.mean():.2f}, std: {y_train.std():.2f}')
print(f'Test — 평균: {y_test.mean():.2f}, std: {y_test.std():.2f}')


훈련 세트: 336개 행정동
테스트 세트: 84개 행정동

Train — 평균: 30.04, std: 20.52
Test — 평균: 30.84, std: 22.65


In [ ]:
CV_SPLITS = 5

# CV 전략 1: 표준 KFold
cv_kfold  = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# CV 전략 2: Y 5분위 층화 StratifiedKFold (회귀용 커스텀)
cv_skfold = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print('=' * 62)
print('[CV 전략 설명]')
print('=' * 62)
desc = """
CV-1  KFold(5)
  • 5겹 무작위 분할 — 기본 회귀 CV 기준선
  • 총 5번 학습, 평균 성능 & 표준편차 산출

CV-2  StratifiedKFold(5)  on  Y-quintiles
  • Y(요양수요가속도)를 5분위수로 구간화 → 5개 strata
  • 각 fold에 '고가속/저가속' 행정동이 균등 배분
  • 소수 극단 지역(이상치)이 특정 fold에 몰리는 편향 방지
  • 회귀 Y를 분류 strata로 변환 → StratifiedKFold.split(X, y_bins)
"""
print(desc)


---
## 3. 헬퍼 함수 및 파이프라인


In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def mape_safe(y_true, y_pred, threshold=0.5):
    """Y가 로그수익률(음수 포함)이므로 |y_true| < threshold 샘플 제외"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.abs(y_true) >= threshold
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


rmse_scorer = make_scorer(rmse, greater_is_better=False)
mape_scorer = make_scorer(mape_safe, greater_is_better=False)

CV_SCORING = {
    'RMSE': rmse_scorer,
    'MAE' : 'neg_mean_absolute_error',
    'R2'  : 'r2',
    'MAPE': mape_scorer,
}

print('평가 지표 정의 완료 ✓')
print('  RMSE — 루트 평균 제곱 오차')
print('  MAE  — 평균 절대 오차')
print('  R²   — 결정계수 (설명력)')
print('  MAPE — 절대 백분율 오차 (|Y|≥0.5 샘플만 집계)')


In [ ]:
def run_stratified_cv(estimator, X, y, y_bins, n_splits=5, random_state=42):
    """
    회귀 문제에서 Y를 분위수 구간으로 층화하여 StratifiedKFold 수행.
    cross_validate는 연속형 y에 StratifiedKFold를 직접 지원하지 않으므로
    fold 인덱스를 수동 생성 후 각 fold별 train/val 점수를 계산한다.
    """
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    res = {k: {'train': [], 'test': []} for k in ['RMSE', 'MAE', 'R2', 'MAPE']}

    for train_idx, val_idx in cv.split(X, y_bins):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

        est = clone(estimator)
        est.fit(X_tr, y_tr)
        p_tr = est.predict(X_tr)
        p_va = est.predict(X_va)

        res['RMSE']['train'].append(rmse(y_tr, p_tr))
        res['RMSE']['test'].append(rmse(y_va, p_va))
        res['MAE']['train'].append(mean_absolute_error(y_tr, p_tr))
        res['MAE']['test'].append(mean_absolute_error(y_va, p_va))
        res['R2']['train'].append(r2_score(y_tr, p_tr))
        res['R2']['test'].append(r2_score(y_va, p_va))
        res['MAPE']['train'].append(mape_safe(y_tr.values, p_tr))
        res['MAPE']['test'].append(mape_safe(y_va.values, p_va))

    return {f'{split}_{metric}': np.array(res[metric][split])
            for metric in res for split in ['train', 'test']}


print('run_stratified_cv() 정의 완료 ✓')


In [ ]:
def build_pipeline(model, scale=True):
    steps = []
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    return Pipeline(steps)


MODELS = {
    'Ridge'       : build_pipeline(Ridge(alpha=1.0), scale=True),
    'Lasso'       : build_pipeline(Lasso(alpha=0.1, max_iter=10000), scale=True),
    'DecisionTree': build_pipeline(DecisionTreeRegressor(random_state=RANDOM_STATE), scale=False),
    'RandomForest': build_pipeline(
        RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1), scale=False),
    'XGBoost'     : build_pipeline(
        XGBRegressor(n_estimators=100, random_state=RANDOM_STATE, verbosity=0, n_jobs=-1), scale=False),
}

print('모델 파이프라인:')
for name, pipe in MODELS.items():
    steps_str = ' → '.join(s[0] for s in pipe.steps)
    print(f'  {name:<15}: {steps_str}')


---
## 4. 기본 모델 비교 (CV 없이 — Train/Test 직접 평가)

CV 없이 **8:2 분할 데이터**를 기본 하이퍼파라미터 그대로 학습·평가하여
각 모델의 출발점 성능과 과적합 경향을 먼저 확인한다.


In [ ]:
print('기본 모델 Train/Test 평가 중 (기본 하이퍼파라미터)...')

baseline_results = []
baseline_preds   = {}

for name, pipe in MODELS.items():
    m = clone(pipe)
    m.fit(X_train, y_train)
    p_train = m.predict(X_train)
    p_test  = m.predict(X_test)
    baseline_results.append({
        '모델':        name,
        'Train_RMSE':  rmse(y_train, p_train),
        'Test_RMSE':   rmse(y_test,  p_test),
        'Train_R2':    r2_score(y_train, p_train),
        'Test_R2':     r2_score(y_test,  p_test),
        'Test_MAE':    mean_absolute_error(y_test, p_test),
        'Test_MAPE':   mape_safe(y_test.values, p_test),
        'Overfit_Gap': rmse(y_test, p_test) - rmse(y_train, p_train),
    })
    baseline_preds[name] = {'train': p_train, 'test': p_test}
    print(f'  {name} ✓')

df_baseline = pd.DataFrame(baseline_results).sort_values('Test_RMSE')
print()
print('[기본 모델 비교 (CV 없이) — 기본 하이퍼파라미터]')
print('=' * 75)
print(df_baseline.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


In [ ]:
sorted_base = df_baseline['모델'].tolist()
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
x = np.arange(len(sorted_base))

for ax, (tr_col, te_col, title) in zip(axes, [
    ('Train_RMSE', 'Test_RMSE', 'Train vs. Test RMSE'),
    ('Train_R2',   'Test_R2',   'Train vs. Test R²'),
    ('Overfit_Gap', None,       '과적합 격차 (Test − Train RMSE)'),
]):
    if te_col is None:
        vals = [df_baseline[df_baseline['모델']==n]['Overfit_Gap'].values[0] for n in sorted_base]
        colors_g = ['#ef4444' if v > 3 else '#34d399' for v in vals]
        ax.bar(sorted_base, vals, color=colors_g, alpha=0.85, edgecolor='white')
        ax.axhline(0, color='white', lw=1, ls='--')
        for i, v in enumerate(vals):
            ax.text(i, v + 0.1, f'{v:+.2f}', ha='center', fontsize=9,
                    color='white', fontweight='bold')
    else:
        tr_v = [df_baseline[df_baseline['모델']==n][tr_col].values[0] for n in sorted_base]
        te_v = [df_baseline[df_baseline['모델']==n][te_col].values[0] for n in sorted_base]
        ax.bar(x - 0.2, tr_v, 0.38, label='Train', color='#3b82f6', alpha=0.8, edgecolor='white')
        ax.bar(x + 0.2, te_v, 0.38, label='Test',  color='#ef4444', alpha=0.8, edgecolor='white')
        ax.set_xticks(x)
        ax.legend(fontsize=9)
        if 'R2' in tr_col:
            ax.axhline(0, color='#f87171', lw=1, ls='--')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticklabels(sorted_base, rotation=20, ha='right')

plt.suptitle('기본 모델 비교 — CV 없이 Train/Test 직접 평가 (기본 하이퍼파라미터)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---
## 5. 교차검증 성능 비교 (KFold-5 / StratifiedKFold-5)

각 모델을 **KFold-5 / StratifiedKFold-5** 두 전략으로 교차검증한다.  
`return_train_score=True`로 **훈련-검증 격차(과적합 진단)**도 함께 확인한다.


In [ ]:
# KFold: cross_validate 사용
print('CV 실행 중... (KFold-5)')

baseline_kfold = {}

for name, pipe in MODELS.items():
    baseline_kfold[name] = cross_validate(
        pipe, X_train, y_train,
        cv=cv_kfold, scoring=CV_SCORING,
        return_train_score=True, n_jobs=-1
    )
    print(f'  {name} ✓')

print('완료!')


In [ ]:
# StratifiedKFold: Y 분위수 층화 — 회귀용 커스텀 함수 사용
print('기본 CV 실행 중... (StratifiedKFold - Y 5분위 층화)')

baseline_skfold = {}

for name, pipe in MODELS.items():
    baseline_skfold[name] = run_stratified_cv(
        pipe, X_train, y_train,
        y_bins=y_train_bins,
        n_splits=CV_SPLITS,
        random_state=RANDOM_STATE
    )
    print(f'  {name} ✓')

print('완료!')


In [ ]:
# 결과 집계 유틸리티
def get_scores(cv_dict, split='test', source='cross_validate'):
    """cross_validate / run_stratified_cv 결과를 통일된 형식으로 반환"""
    sign = {'RMSE': -1, 'MAE': -1, 'R2': 1, 'MAPE': -1}
    out  = {}
    for m, s in sign.items():
        if source == 'cross_validate':
            key = f'{split}_{m}'
            arr = s * cv_dict.get(key, np.zeros(1))
        else:  # run_stratified_cv
            key = f'{split}_{m}'
            arr = cv_dict.get(key, np.zeros(1)).copy()
        out[m] = arr
    return out


model_names = list(MODELS.keys())
rows = []

for name in model_names:
    for cv_label, cv_data, src in [
        ('KFold-5',      baseline_kfold[name],  'cross_validate'),
        ('StratKFold-5', baseline_skfold[name], 'stratified'),
    ]:
        tr = get_scores(cv_data, 'train', src)
        va = get_scores(cv_data, 'test',  src)
        rows.append({
            '모델': name, 'CV전략': cv_label,
            'Train_RMSE' : tr['RMSE'].mean(),
            'Val_RMSE'   : va['RMSE'].mean(),
            'Val_RMSE_std': va['RMSE'].std(),
            'Train_R2'   : tr['R2'].mean(),
            'Val_R2'     : va['R2'].mean(),
            'Val_MAE'    : va['MAE'].mean(),
            'Val_MAPE'   : va['MAPE'].mean(),
            'Overfit_RMSE': va['RMSE'].mean() - tr['RMSE'].mean(),
        })

df_cv = pd.DataFrame(rows)

print('[기본 모델 CV 결과 — Validation 지표]')
print('=' * 92)
print(df_cv[['모델', 'CV전략', 'Val_RMSE', 'Val_RMSE_std',
              'Val_MAE', 'Val_R2', 'Val_MAPE', 'Overfit_RMSE']]
      .to_string(index=False, float_format=lambda x: f'{x:.3f}'))


In [ ]:
# CV 전략 비교 시각화
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
cv_labels  = ['KFold-5', 'StratKFold-5']
palette    = ['#3b82f6', '#10b981']
x = np.arange(len(model_names))
width = 0.35

for ax_i, metric in enumerate(['Val_RMSE', 'Val_R2', 'Val_MAE']):
    ax = axes[ax_i]
    for j, (cv_label, color) in enumerate(zip(cv_labels, palette)):
        sub  = df_cv[df_cv['CV전략'] == cv_label]
        vals = [sub[sub['모델'] == m][metric].values[0] for m in model_names]
        errs = None
        if metric == 'Val_RMSE':
            errs = [sub[sub['모델'] == m]['Val_RMSE_std'].values[0] for m in model_names]
        ax.bar(x + (j - 0.5) * width, vals, width,
               label=cv_label, color=color, alpha=0.85, edgecolor='white',
               yerr=errs, capsize=3,
               error_kw={'ecolor': 'white', 'lw': 1.5} if errs else {})
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=25, ha='right', fontsize=10)
    ax.set_title(f'Val {metric.replace("Val_","")}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    if metric == 'Val_R2':
        ax.axhline(0, color='#f87171', lw=1, ls='--')

plt.suptitle('기본 모델 — KFold-5 vs. StratifiedKFold-5 비교',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 과적합 진단: Train vs. Validation RMSE
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
cv_info = [
    ('KFold-5',      baseline_kfold,  'cross_validate'),
    ('StratKFold-5', baseline_skfold, 'stratified'),
]

for ax_i, (cv_label, cv_data, src) in enumerate(cv_info):
    ax = axes[ax_i]
    tr_means, va_means = [], []
    tr_stds,  va_stds  = [], []
    for name in model_names:
        tr = get_scores(cv_data[name], 'train', src)
        va = get_scores(cv_data[name], 'test',  src)
        tr_means.append(tr['RMSE'].mean()); tr_stds.append(tr['RMSE'].std())
        va_means.append(va['RMSE'].mean()); va_stds.append(va['RMSE'].std())

    xp = np.arange(len(model_names))
    ax.bar(xp - 0.2, tr_means, 0.38, label='Train RMSE',
           color='#3b82f6', alpha=0.8, edgecolor='white',
           yerr=tr_stds, capsize=4, error_kw={'ecolor': 'white', 'lw': 1.5})
    ax.bar(xp + 0.2, va_means, 0.38, label='Val RMSE',
           color='#ef4444', alpha=0.8, edgecolor='white',
           yerr=va_stds, capsize=4, error_kw={'ecolor': 'white', 'lw': 1.5})
    ax.set_xticks(xp)
    ax.set_xticklabels(model_names, rotation=25, ha='right', fontsize=9)
    ax.set_title(f'{cv_label}', fontsize=11, fontweight='bold')
    ax.set_ylabel('RMSE')
    ax.legend(fontsize=8)
    for i, (tr, va) in enumerate(zip(tr_means, va_means)):
        gap = va - tr
        color_g = '#f87171' if gap > 5 else '#34d399'
        ax.text(xp[i], max(tr, va) + 0.3, f'{gap:+.1f}',
                ha='center', fontsize=8, color=color_g, fontweight='bold')

plt.suptitle('과적합 진단 — Train vs. Validation RMSE 격차 (숫자=격차)',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()


---
## 6. 하이퍼파라미터 튜닝 (전 모델 GridSearchCV)

- **Ridge / Lasso** → `GridSearchCV(cv=KFold-5, scoring=RMSE)`
- **Decision Tree** → `GridSearchCV(cv=KFold-5)`
- **Random Forest** → `GridSearchCV(cv=KFold-5)`
- **XGBoost** → `GridSearchCV(cv=KFold-5)`

내부 CV에는 `KFold(5)`를 사용하고, 최종 모델은 전체 훈련 세트로 재학습(`refit=True`) 후 테스트 세트에서 평가한다.


In [ ]:
# Ridge / Lasso GridSearchCV
print('[Ridge GridSearchCV]')
ridge_gs = GridSearchCV(
    build_pipeline(Ridge(), scale=True),
    {'model__alpha': [0.001, 0.01, 0.1, 1, 10, 50, 100, 500, 1000]},
    cv=cv_kfold, scoring=rmse_scorer, refit=True, n_jobs=-1
)
ridge_gs.fit(X_train, y_train)
print(f'  최적 alpha: {ridge_gs.best_params_["model__alpha"]}  |  CV RMSE: {-ridge_gs.best_score_:.4f}')

print('[Lasso GridSearchCV]')
lasso_gs = GridSearchCV(
    build_pipeline(Lasso(max_iter=10000), scale=True),
    {'model__alpha': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10]},
    cv=cv_kfold, scoring=rmse_scorer, refit=True, n_jobs=-1
)
lasso_gs.fit(X_train, y_train)
print(f'  최적 alpha: {lasso_gs.best_params_["model__alpha"]}  |  CV RMSE: {-lasso_gs.best_score_:.4f}')

# alpha vs RMSE 곡선
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, gs, label, color in zip(
    axes, [ridge_gs, lasso_gs], ['Ridge', 'Lasso'], ['#3b82f6', '#10b981']
):
    res = pd.DataFrame(gs.cv_results_)
    alphas = [p['model__alpha'] for p in gs.cv_results_['params']]
    means  = -res['mean_test_score'].values
    stds   =  res['std_test_score'].values
    ax.semilogx(alphas, means, 'o-', color=color, lw=2, label='Val RMSE')
    ax.fill_between(alphas, means - stds, means + stds, alpha=0.2, color=color)
    best_a = gs.best_params_['model__alpha']
    ax.axvline(best_a, color='#f59e0b', ls='--', lw=1.5, label=f'최적 α={best_a}')
    ax.set_title(f'{label} — alpha 탐색', fontsize=12, fontweight='bold')
    ax.set_xlabel('alpha (log scale)')
    ax.set_ylabel('CV RMSE')
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Decision Tree GridSearchCV
print('[Decision Tree GridSearchCV]')
dt_gs = GridSearchCV(
    build_pipeline(DecisionTreeRegressor(random_state=RANDOM_STATE), scale=False),
    {
        'model__max_depth'        : [None, 3, 5, 7, 10],
        'model__min_samples_split': [2, 5, 10, 20],
        'model__min_samples_leaf' : [1, 3, 5, 10],
        'model__max_features'     : [None, 'sqrt', 'log2'],
    },
    cv=cv_kfold, scoring=rmse_scorer, refit=True, n_jobs=-1
)
dt_gs.fit(X_train, y_train)
print(f'  최적 파라미터: {dt_gs.best_params_}')
print(f'  CV RMSE     : {-dt_gs.best_score_:.4f}')

# max_depth별 평균 RMSE
res_dt = pd.DataFrame(dt_gs.cv_results_)
depth_grp = res_dt.groupby('param_model__max_depth')['mean_test_score'].agg(['mean','std'])
depth_grp.index = depth_grp.index.astype(str).fillna('None')
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(depth_grp.index, -depth_grp['mean'], color='#8b5cf6', alpha=0.8, edgecolor='white',
       yerr=depth_grp['std'], capsize=5, error_kw={'ecolor': 'white', 'lw': 1.5})
ax.set_title('Decision Tree — max_depth별 CV RMSE', fontsize=12, fontweight='bold')
ax.set_xlabel('max_depth')
ax.set_ylabel('CV RMSE')
plt.tight_layout()
plt.show()


In [ ]:
# Random Forest GridSearchCV
print('[Random Forest GridSearchCV]')
rf_gs = GridSearchCV(
    build_pipeline(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), scale=False),
    {
        'model__n_estimators'     : [100, 200, 300],
        'model__max_depth'        : [None, 5, 10],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf' : [1, 2, 4],
        'model__max_features'     : ['sqrt', 'log2'],
    },
    cv=cv_kfold, scoring=rmse_scorer, refit=True, n_jobs=-1, verbose=1
)
rf_gs.fit(X_train, y_train)
print(f'  최적 파라미터: {rf_gs.best_params_}')
print(f'  CV RMSE     : {-rf_gs.best_score_:.4f}')


In [ ]:
# XGBoost GridSearchCV
print('[XGBoost GridSearchCV]')
xgb_gs = GridSearchCV(
    build_pipeline(
        XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=-1), scale=False),
    {
        'model__n_estimators'    : [100, 200, 300],
        'model__max_depth'       : [3, 5, 7],
        'model__learning_rate'   : [0.05, 0.1, 0.2],
        'model__subsample'       : [0.8, 1.0],
        'model__colsample_bytree': [0.8, 1.0],
        'model__reg_alpha'       : [0, 0.1, 1],
        'model__reg_lambda'      : [1, 5],
    },
    cv=cv_kfold, scoring=rmse_scorer, refit=True, n_jobs=-1, verbose=1
)
xgb_gs.fit(X_train, y_train)
print(f'  최적 파라미터: {xgb_gs.best_params_}')
print(f'  CV RMSE     : {-xgb_gs.best_score_:.4f}')


In [ ]:
# 튜닝 후 모델 집합
tuned_models = {
    'Ridge'       : ridge_gs.best_estimator_,
    'Lasso'       : lasso_gs.best_estimator_,
    'DecisionTree': dt_gs.best_estimator_,
    'RandomForest': rf_gs.best_estimator_,
    'XGBoost'     : xgb_gs.best_estimator_,
}

tuning_summary = pd.DataFrame([
    {'모델': 'Ridge',        '방식': 'GridSearchCV', '조합수':   9, 'CV_RMSE': -ridge_gs.best_score_},
    {'모델': 'Lasso',        '방식': 'GridSearchCV', '조합수':   9, 'CV_RMSE': -lasso_gs.best_score_},
    {'모델': 'DecisionTree', '방식': 'GridSearchCV', '조합수': '전수', 'CV_RMSE': -dt_gs.best_score_},
    {'모델': 'RandomForest', '방식': 'GridSearchCV', '조합수': 162, 'CV_RMSE': -rf_gs.best_score_},
    {'모델': 'XGBoost',      '방식': 'GridSearchCV', '조합수': 288, 'CV_RMSE': -xgb_gs.best_score_},
]).sort_values('CV_RMSE')

print('[하이퍼파라미터 튜닝 후 CV RMSE 비교 (전 모델 GridSearchCV)]')
print(tuning_summary.to_string(index=False))


---
## 7. 최종 모델 평가 (Holdout Test Set)


In [ ]:
print('[최종 모델 — Holdout Test Set 평가]')
print('=' * 65)

test_results = []
predictions  = {}

for name, model in tuned_models.items():
    p_train = model.predict(X_train)
    p_test  = model.predict(X_test)

    test_results.append({
        '모델'       : name,
        'Train_RMSE' : rmse(y_train, p_train),
        'Test_RMSE'  : rmse(y_test,  p_test),
        'Train_R2'   : r2_score(y_train, p_train),
        'Test_R2'    : r2_score(y_test,  p_test),
        'Test_MAE'   : mean_absolute_error(y_test, p_test),
        'Test_MAPE'  : mape_safe(y_test.values, p_test),
        'Overfit_Gap': rmse(y_test, p_test) - rmse(y_train, p_train),
    })
    predictions[name] = {'train': p_train, 'test': p_test}

df_test = pd.DataFrame(test_results).sort_values('Test_RMSE')
print(df_test.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

BEST_MODEL_NAME = df_test.iloc[0]['모델']
print(f'\n★ 최적 모델: {BEST_MODEL_NAME}')
print(f'  Test RMSE={df_test.iloc[0]["Test_RMSE"]:.3f}  '
      f'Test R²={df_test.iloc[0]["Test_R2"]:.3f}  '
      f'Test MAE={df_test.iloc[0]["Test_MAE"]:.3f}')


In [ ]:
# 최종 비교 시각화
sorted_names = df_test['모델'].tolist()
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
x = np.arange(len(sorted_names))

for ax, (metric_tr, metric_te, title) in zip(axes, [
    ('Train_RMSE', 'Test_RMSE', 'Train vs. Test RMSE'),
    ('Train_R2',   'Test_R2',   'Train vs. Test R²'),
    ('Overfit_Gap', None,       '과적합 격차 (Test - Train RMSE)'),
]):
    if metric_te is None:
        vals = [df_test[df_test['모델'] == n][metric_tr].values[0] for n in sorted_names]
        colors_g = ['#ef4444' if v > 3 else '#34d399' for v in vals]
        ax.bar(sorted_names, vals, color=colors_g, alpha=0.85, edgecolor='white')
        ax.axhline(0, color='white', lw=1, ls='--')
        for i, v in enumerate(vals):
            ax.text(i, v + 0.1, f'{v:+.2f}', ha='center', fontsize=9,
                    color='white', fontweight='bold')
    else:
        tr_v = [df_test[df_test['모델'] == n][metric_tr].values[0] for n in sorted_names]
        te_v = [df_test[df_test['모델'] == n][metric_te].values[0] for n in sorted_names]
        ax.bar(x - 0.2, tr_v, 0.38, label='Train', color='#3b82f6', alpha=0.8, edgecolor='white')
        ax.bar(x + 0.2, te_v, 0.38, label='Test',  color='#ef4444', alpha=0.8, edgecolor='white')
        ax.set_xticks(x)
        ax.legend(fontsize=9)
        if 'R2' in metric_tr:
            ax.axhline(0, color='#f87171', lw=1, ls='--')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticklabels(sorted_names, rotation=20, ha='right')

plt.suptitle('Track A — 튜닝 후 최종 모델 비교', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---
## 8. 학습 곡선 (Learning Curve)


In [ ]:
fig, axes = plt.subplots(1, len(tuned_models), figsize=(20, 4.5), sharey=False)

for ax, (name, model) in zip(axes, tuned_models.items()):
    train_sizes, train_sc, val_sc = learning_curve(
        model, X_train, y_train,
        train_sizes=np.linspace(0.15, 1.0, 10),
        cv=KFold(5, shuffle=True, random_state=RANDOM_STATE),
        scoring=rmse_scorer, n_jobs=-1
    )
    tr_mean = -train_sc.mean(axis=1); tr_std = train_sc.std(axis=1)
    va_mean = -val_sc.mean(axis=1);   va_std = val_sc.std(axis=1)

    ax.plot(train_sizes, tr_mean, 'o-', color='#3b82f6', lw=2, label='Train RMSE')
    ax.fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std, alpha=0.2, color='#3b82f6')
    ax.plot(train_sizes, va_mean, 's--', color='#ef4444', lw=2, label='Val RMSE')
    ax.fill_between(train_sizes, va_mean - va_std, va_mean + va_std, alpha=0.2, color='#ef4444')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('훈련 샘플 수')
    ax.set_ylabel('RMSE')
    ax.legend(fontsize=8)

plt.suptitle('학습 곡선 — 훈련 샘플 수에 따른 Train/Val RMSE\n'
             '(갭이 좁아질수록 데이터 증가 효과적)',
             fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()


---
## 9. 최선 모델 심층 분석


In [ ]:
best_model   = tuned_models[BEST_MODEL_NAME]
y_pred_test  = predictions[BEST_MODEL_NAME]['test']
residuals    = y_test.values - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 잔차 분포
axes[0].hist(residuals, bins=25, color='#8b5cf6', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='#f59e0b', lw=2, ls='--', label='잔차=0')
axes[0].axvline(residuals.mean(), color='#f87171', lw=1.5, ls='--',
                label=f'평균={residuals.mean():.2f}')
axes[0].set_title(f'{BEST_MODEL_NAME} — 잔차 분포', fontsize=12, fontweight='bold')
axes[0].set_xlabel('잔차 (실제 - 예측)')
axes[0].legend(fontsize=9)

# 예측 vs 실제
min_v = min(y_test.min(), y_pred_test.min())
max_v = max(y_test.max(), y_pred_test.max())
sc = axes[1].scatter(y_test, y_pred_test, alpha=0.6, s=30,
                     c=np.abs(residuals), cmap='RdYlGn_r', edgecolors='none')
axes[1].plot([min_v, max_v], [min_v, max_v], 'w--', lw=1.5, label='완벽한 예측')
plt.colorbar(sc, ax=axes[1], shrink=0.8, label='|잔차|')
axes[1].set_title('실제 vs. 예측', fontsize=12, fontweight='bold')
axes[1].set_xlabel('실제 요양수요가속도')
axes[1].set_ylabel('예측 요양수요가속도')
axes[1].legend(fontsize=9)

# 잔차 vs 예측 (이분산성)
axes[2].scatter(y_pred_test, residuals, alpha=0.6, s=30,
                c=np.abs(residuals), cmap='RdYlGn_r', edgecolors='none')
axes[2].axhline(0, color='#f59e0b', lw=2, ls='--')
axes[2].set_title('잔차 vs. 예측값 (이분산성 확인)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('예측값')
axes[2].set_ylabel('잔차')

plt.suptitle(f'잔차 분석 — {BEST_MODEL_NAME}', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'잔차 통계:')
print(f'  평균: {residuals.mean():.3f}  std: {residuals.std():.3f}')
print(f'  최대 과대예측: {residuals.min():.2f}  최대 과소예측: {residuals.max():.2f}')


In [ ]:
try:
    import shap

    # Pipeline에서 실제 모델 및 변환된 X 추출
    if hasattr(best_model, 'named_steps'):
        step_keys  = list(best_model.named_steps.keys())
        core_model = best_model.named_steps[step_keys[-1]]
        if 'scaler' in best_model.named_steps:
            X_tr_t = pd.DataFrame(
                best_model.named_steps['scaler'].transform(X_train), columns=FEAT_COLS)
            X_te_t = pd.DataFrame(
                best_model.named_steps['scaler'].transform(X_test),  columns=FEAT_COLS)
        else:
            X_tr_t, X_te_t = X_train.copy(), X_test.copy()
    else:
        core_model = best_model
        X_tr_t, X_te_t = X_train.copy(), X_test.copy()

    model_type = type(core_model).__name__
    if any(t in model_type for t in ['XGB', 'Forest', 'Tree']):
        explainer   = shap.TreeExplainer(core_model)
        shap_values = explainer.shap_values(X_te_t)
    else:
        explainer   = shap.LinearExplainer(core_model, X_tr_t)
        shap_values = explainer.shap_values(X_te_t)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 전역 중요도 막대
    mean_abs = np.abs(shap_values).mean(axis=0)
    shap_df  = pd.DataFrame({'피처': FEAT_COLS, 'Mean |SHAP|': mean_abs}).sort_values('Mean |SHAP|')
    axes[0].barh(shap_df['피처'], shap_df['Mean |SHAP|'],
                 color=plt.cm.RdBu_r(np.linspace(0.2, 0.8, len(FEAT_COLS))),
                 edgecolor='white', alpha=0.85)
    axes[0].set_title(f'{BEST_MODEL_NAME} — SHAP 전역 중요도', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Mean |SHAP value|')

    # Beeswarm
    plt.sca(axes[1])
    shap.summary_plot(shap_values, X_te_t, feature_names=FEAT_COLS,
                      show=False, plot_type='dot', max_display=9)
    axes[1].set_title('SHAP Beeswarm — 피처 영향 방향', fontsize=12, fontweight='bold')

    plt.suptitle(f'SHAP 피처 중요도 분석 ({BEST_MODEL_NAME})',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    print('\n[SHAP 피처 중요도 순위]')
    print(shap_df.sort_values('Mean |SHAP|', ascending=False).to_string(index=False))

except Exception as e:
    print(f'SHAP 오류: {e}')
    if hasattr(core_model, 'feature_importances_'):
        fi_df = pd.DataFrame({'피처': FEAT_COLS,
                              '중요도': core_model.feature_importances_})\
                  .sort_values('중요도', ascending=False)
        print('\n[sklearn Feature Importance 대체]')
        print(fi_df.to_string(index=False))


---
## 10. 수요 가속도 상위 / 하위 행정동 분석


In [ ]:
y_pred_all = best_model.predict(X_all)

df_result = meta.copy()
df_result['실제_요양수요가속도'] = y_all.values
df_result['예측_요양수요가속도'] = y_pred_all
df_result['잔차']               = y_all.values - y_pred_all

top10 = df_result.nlargest(10,  '실제_요양수요가속도')[
    ['행정동명', '자치구명', '실제_요양수요가속도', '예측_요양수요가속도', '잔차']
].reset_index(drop=True)
bot10 = df_result.nsmallest(10, '실제_요양수요가속도')[
    ['행정동명', '자치구명', '실제_요양수요가속도', '예측_요양수요가속도', '잔차']
].reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, data, title, color in zip(
    axes,
    [top10, bot10],
    ['요양수요가속도 상위 10개 (고위험)', '요양수요가속도 하위 10개 (저위험)'],
    ['#ef4444', '#3b82f6']
):
    labels = data['자치구명'] + '\n' + data['행정동명']
    xp = np.arange(len(labels))
    ax.bar(xp - 0.2, data['실제_요양수요가속도'], 0.38, label='실제값',
           color=color, alpha=0.85, edgecolor='white')
    ax.bar(xp + 0.2, data['예측_요양수요가속도'], 0.38, label='예측값',
           color='#94a3b8', alpha=0.7, edgecolor='white')
    ax.set_xticks(xp)
    ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax.axhline(0, color='white', lw=1, ls='--')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('요양수요가속도')
    ax.legend(fontsize=9)
plt.suptitle(f'극단 지역 실제 vs. 예측 ({BEST_MODEL_NAME})',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('[수요가속도 상위 10개]')
print(top10.to_string(index=False, float_format='{:.2f}'.format))
print()
print('[수요가속도 하위 10개]')
print(bot10.to_string(index=False, float_format='{:.2f}'.format))


In [ ]:
top10_idx = df_result.nlargest(10,  '실제_요양수요가속도').index
bot10_idx = df_result.nsmallest(10, '실제_요양수요가속도').index

profile = pd.DataFrame({
    '상위10(고위험)': X_all.loc[top10_idx].mean(),
    '하위10(저위험)': X_all.loc[bot10_idx].mean(),
    '전체평균':       X_all.mean(),
})

# Min-Max 정규화
p_min = profile.min(axis=1)
p_max = profile.max(axis=1)
profile_norm = profile.sub(p_min, axis=0).div((p_max - p_min).replace(0, 1), axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(FEAT_COLS))
for j, (col, color) in enumerate(zip(
    ['상위10(고위험)', '하위10(저위험)', '전체평균'],
    ['#ef4444', '#3b82f6', '#94a3b8']
)):
    ax.bar(x + (j - 1) * 0.27, profile_norm[col], 0.27,
           label=col, color=color, alpha=0.8, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(FEAT_COLS, rotation=30, ha='right', fontsize=10)
ax.set_title('상위 / 하위 10개 행정동 피처 프로파일 (Min-Max 정규화)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('정규화 값 (0~1)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('[원본 값 프로파일]')
print(profile.to_string(float_format='{:.3f}'.format))


---
## 11. 최종 결과 요약


In [ ]:
print('=' * 78)
print('Track A — 최종 모델 성능 요약 (KFold-5 / StratKFold-5 Validation + Holdout Test)')
print('=' * 78)

final_rows = []
for name in model_names:
    t_row   = df_test[df_test['모델'] == name].iloc[0]
    kf_rmse = -baseline_kfold[name]['test_RMSE'].mean()
    kf_std  =  baseline_kfold[name]['test_RMSE'].std()
    kf_r2   =  baseline_kfold[name]['test_R2'].mean()
    final_rows.append({
        '모델'              : name,
        'KFold Val RMSE'   : f'{kf_rmse:.3f}±{kf_std:.3f}',
        'KFold Val R²'     : f'{kf_r2:.3f}',
        'Test RMSE'        : f'{t_row["Test_RMSE"]:.3f}',
        'Test R²'          : f'{t_row["Test_R2"]:.3f}',
        'Test MAE'         : f'{t_row["Test_MAE"]:.3f}',
        'Test MAPE(%)'     : f'{t_row["Test_MAPE"]:.1f}',
        '과적합갭'          : f'{t_row["Overfit_Gap"]:+.3f}',
    })

print(pd.DataFrame(final_rows).to_string(index=False))
print()
print(f'★ 최적 모델: {BEST_MODEL_NAME}')
print()
print('[CV 전략 인사이트]')
print('  KFold-5      : 표준 기준선')
print('  StratKFold-5 : Y 분위수 층화 → fold별 분포 균형 확보')
print()
print('[주요 피처 발견]')
print('  독거노인비율  (r=+0.35)  — Y와 가장 높은 양의 상관')
print('  고령화율_변화 (r=+0.24)  — 증가 속도 트렌드 반영')
print('  비율_75-79세  (r=-0.29)  — 음의 상관: 청장년 고령층 많을수록 가속도 낮음')


In [ ]:
# 결과 저장
df_result['모델명'] = BEST_MODEL_NAME
df_result.to_csv('data/trackA_predictions.csv', index=False, encoding='utf-8-sig')
print(f'예측 결과 저장 완료: data/trackA_predictions.csv  ({len(df_result)}행)')
df_result.head()
